# Analysis

In [18]:
import pandas as pd

race_results = pd.read_csv("data/raw/f1db-races-race-results.csv")
races = pd.read_csv("data/raw/f1db-races.csv")
quali_results = pd.read_csv("data/raw/f1db-races-qualifying-results.csv")
circuits = pd.read_csv("data/raw/f1db-circuits.csv")


races = races[races["year"].between(2010, 2026)]
race_results = race_results[race_results["year"].between(2010, 2026)]
quali_results = quali_results[quali_results["year"].between(2010, 2026)]

# One row per race
# Subset focuses only on the id -> race. Sum then adds up all the True values from duplicated() which only occur if an id is there for more than one time
print("Duplicate races:", races.duplicated(subset=["id"]).sum())

# One row per driver and race
print("Duplicate race results:", race_results.duplicated(subset=["raceId", "driverId"]).sum())

print("Duplicate qualifying results:", quali_results.duplicated(subset=["raceId", "driverId"]).sum())



Duplicate races: 0
Duplicate race results: 0
Duplicate qualifying results: 0


/var/folders/y6/yvr9h5p110qc_zbybx4w5p680000gn/T/ipykernel_34687/2600749808.py:3: DtypeWarning: Columns (31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  race_results = pd.read_csv("data/raw/f1db-races-race-results.csv")


- Races: Checks whether a race ID appears more than once.
- Race results: Checks whether a driver appears more than once in the same race.
- Qualifying results: Checks whether a driver appears more than once in the same qualifying session.

Especially important for new data updates

## Merge data sets

In [19]:
# how="inner" means that only for available ids of race_results, the values of race get joined (not too important here since the data is already very cleanly prepared)
# validate: raceId could be contained multiple times but Id must only appear once, else a MergeError would be triggered
# drop: drops id since we have raceId and id which are the same

model_data = race_results.merge(
    races[[
        "id",
        "date",
        "circuitId",
        "circuitType",
        "courseLength",
        "turns",
        "scheduledLaps"]],
    left_on="raceId",
    right_on="id",
    how="inner",
    validate="many_to_one"
).drop(columns="id")


# on: assigns to every driver the qualifying variables to the according race
# left: all the data of model_data stays, where the suiting parts from quali results get added
# rename is needed, since positionNumber is in the model_data used for the final position

model_data = model_data.merge(
    quali_results[[
        "raceId",
        "driverId",
        "q1Millis",
        "q2Millis",
        "q3Millis",
    ]],
    on=["raceId", "driverId"],
    how="left",
    validate="one_to_one"
)


## How to deal with missing values in gridPositionNumber

In [20]:
print(model_data["gridPositionText"].unique())

gridPositionNumber_nan = model_data[model_data["gridPositionText"].isna()]

['3' '2' '4' '1' '5' '7' '8' '6' '12' '11' '9' '10' '18' '13' '21' '15'
 '20' '14' 'PL' '19' '17' '16' '22' '24' '23' nan]


All the nan values are because of DNS or DNQ, thus they should be excluded. PL stands for "pit lane", meaning that the drivers had to start from the pit lane, which is more or less euqivalent as starting from the last position. Since the field hasn't always the same size, it can't be just set to 20.

In [21]:
model_data = model_data[model_data["gridPositionText"].notna()]

# transform("size"): size counts rows per group, where raceId is treated as the group

model_data["fieldSize"] = model_data.groupby("raceId")["driverId"].transform("size")


# .loc searches for the column gridPositionText with value "PL" and takes the related gridPositionNumber

model_data.loc[ model_data["gridPositionText"] == "PL", "gridPositionNumber"] = model_data["fieldSize"]

## Exlude and add features

In [22]:
columns_needed = [
    "raceId",
    "fieldSize",
    "year",
    "round",
    "date",
    "driverId",
    "constructorId",
    "circuitId",
    "circuitType",
    "courseLength",
    "turns",
    "scheduledLaps",
    "gridPositionNumber",
    "gridPositionText",
    "qualificationPositionNumber",
    "q1Millis",
    "q2Millis",
    "q3Millis",
    "positionDisplayOrder",
    "positionText",
    "reasonRetired"
]


model_data = model_data[columns_needed].copy()

In [23]:
model_data["qualifying_fraction"] = (
    (model_data["qualificationPositionNumber"] - 1)
    / (model_data["fieldSize"] - 1)
)

model_data["grid_fraction"] = (
    (model_data["gridPositionNumber"] - 1)
    / (model_data["fieldSize"] - 1)
)

model_data["finish_fraction"] = (
    (model_data["positionDisplayOrder"] - 1)
    / (model_data["fieldSize"] - 1)
)

Normalizing positions makes races with different field sizes comparable: finishing 20th out of 20 and 20th out of 22.

#### 5 race average

In [24]:
# transform: applies a computation on every group and returns its results back to the original row.
# shift(1): shifts all values one rows downwards, such that the result only gets included in the next races last five average
# rolling(5): uses five consecutive values
# min_periods says that the value should also be computed if only 4 instead of 5 values are available

model_data["driver_finish_5"] = (
    model_data.groupby("driverId")["finish_fraction"]
    .transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).mean()
    )
)


#### 5 quali average

In [25]:
model_data["driver_quali_5"] = (
    model_data.groupby("driverId")["qualifying_fraction"]
    .transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).mean()
    )
)

#### Quali deviation feature

In [26]:
model_data["quali_vs_recent"] = (
    model_data["qualifying_fraction"]
    - model_data["driver_quali_5"]
)

#### Failure rate

In [27]:
model_data["failure"] = (model_data["positionText"].eq("DNF") | model_data["reasonRetired"].notna())

# .eq("DNF") is the same as == "DNF"
# .notna() gives back True, if a value isn't missing. (Everything that is missing in reasonRetiered hasn't retiered -> should return False)

model_data["failure_rate"] = (model_data.groupby("driverId")["failure"]
                                .transform(lambda failrate: failrate.shift(1).rolling(10, min_periods=5).mean()))

# mean values True and False as 0 and 1 leading to e.g. 0.4 -> 40%

#### Adding other average historical features

In [28]:
for window in [3, 5, 10, 20]:
    for source, name in [("finish_fraction", "finish"), ("qualifying_fraction", "quali"), ("failure", "failure_rate")]:
        model_data[f"driver_{name}_{window}"] = (model_data.groupby("driverId")[source]
            .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean()))

This adds the average values not of the features computed above also for other race windows. i.e. 3, 5, 10 and 20 races

#### Standard deviations: consistance in races

In [ ]:
for window in [5, 10]:
    model_data[f"driver_std_{window}"] = (
        model_data.groupby("driverId")["finish_fraction"].transform(lambda x: x.shift(1).rolling(window, min_periods=2).std())
    )